# Real vs Fake Face Classifier

Trains a real-vs-fake face classifier using the modular `src/` package.
Saves trained model weights to `models/` and result PNGs (curves + prediction grid) to `results/`.

Run this notebook from the project root (so `src/` is importable).

## 1. Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.append("src")

import torch
from torchvision import transforms

import data_setup, engine, model_builder, utils, download_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Kaggle credentials

Set these before downloading. In Colab, prefer **Secrets** (key icon in sidebar)
over typing your key directly into a cell.

In [ ]:
import os

# Option A: Colab secrets (recommended)
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Option B: set directly (don't commit this notebook with real values filled in)
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'

## 3. Download & extract dataset

In [ ]:
data_path = download_data.download_kaggle_dataset(
    dataset="ajaysonicu/deepfakefusion-399k-realfake-faces",
    dest_dir="data",
)

In [ ]:
from pathlib import Path

data_root = Path("data/deepfakefusion_v2_sixsource_dataset")
train_dir = data_root / "train"
val_dir = data_root / "val"
test_dir = data_root / "test"

for split_dir in [train_dir, val_dir, test_dir]:
    for dirpath, dirnames, filenames in os.walk(split_dir):
        if filenames:
            print(f"{dirpath}: {len(filenames)} images")

## 4. Build DataLoaders

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32

data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
])

train_dataloader, val_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    val_dir=val_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE,
)

print(f"Classes: {class_names}")
img, label = next(iter(train_dataloader))
print(f"Batch shape: {img.shape}, labels shape: {label.shape}")

## 5. Choose and train a model

Set `MODEL_TYPE` to `"cnn"`, `"resnet18"`, or `"efficientnet"`.

In [ ]:
MODEL_TYPE = "efficientnet"  # "cnn" | "resnet18" | "efficientnet"
EPOCHS = 5
LR = 1e-3

if MODEL_TYPE == "cnn":
    model = model_builder.TinyVGG(output_shape=len(class_names), image_size=IMAGE_SIZE).to(device)
elif MODEL_TYPE == "resnet18":
    model = model_builder.create_resnet18(output_shape=len(class_names), device=device)
else:
    model = model_builder.create_efficientnet_b0(output_shape=len(class_names), device=device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=LR)

In [ ]:
results = engine.train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    device=device,
)

## 6. Evaluate on the test set

In [ ]:
test_loss, test_acc = engine.evaluate_model(model, test_dataloader, loss_fn, device)

## 7. Save results — curves PNG, prediction grid PNG, model weights

In [ ]:
utils.plot_curves(results, save_path=f"results/{MODEL_TYPE}_curves.png")

In [ ]:
utils.save_prediction_grid(
    model, test_dataloader.dataset, class_names, device,
    save_path=f"results/{MODEL_TYPE}_predictions.png",
)

In [ ]:
utils.save_model(model, target_dir="models", model_name=f"{MODEL_TYPE}_model.pth")

## Next steps

- Try the other two `MODEL_TYPE` options and compare `results/*_curves.png`.
- Or run the same pipeline from the command line:
  ```bash
  python src/train.py --model resnet18 --epochs 5
  ```